In [2]:
import requests
import pandas as pd

In [ ]:
import requests
import pandas as pd

# Kolding coordinates
lat, lon = 55.49, 9.47
start_date = "2026-02-01"
end_date = "2026-05-20" # Using today's date

url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": lat,
    "longitude": lon,
    "start_date": start_date,
    "end_date": end_date,
    # "hourly": "temperature_2m,apparent_temperature,wind_speed_10m,precipitation,sunshine_duration",
    "daily": "temperature_2m_max,temperature_2m_min,temperature_2m_mean,apparent_temperature_max,wind_speed_10m_max,sunshine_duration,rain_sum,snowfall_sum,precipitation_sum",
    "timezone": "Europe/Berlin"
}

response = requests.get(url, params=params)

if response.status_code == 200:
    data = response.json()
    
    df_weather = pd.DataFrame(data['daily'])    
    
    print(df_weather.head(10).to_string(index=False))
else:
    print(f"Failed! Status: {response.status_code}")

      time  temperature_2m_max  temperature_2m_min  temperature_2m_mean  apparent_temperature_max  wind_speed_10m_max  sunshine_duration  rain_sum  snowfall_sum  precipitation_sum
2026-02-01                -3.1                -5.1                 -4.3                      -9.4                25.6               0.00       0.0          0.00                0.0
2026-02-02                -3.1                -5.9                 -4.7                     -10.6                29.7           17171.77       0.0          0.00                0.0
2026-02-03                -0.7                -3.9                 -2.4                      -8.6                32.3           28480.37       0.0          0.00                0.0
2026-02-04                -1.1                -3.2                 -2.1                      -7.7                32.6               0.00       0.1          0.00                0.1
2026-02-05                -1.2                -2.2                 -1.7                      -7.6   

In [15]:
import requests
import pandas as pd

station_id = "06119"  # Kolding
start_date = "2026-02-01T00:00:00Z"
end_date = "2026-05-20T00:00:00Z"
url = "https://opendataapi.dmi.dk/v2/climateData/collections/stationValue/items"

parameters = ["max_temp", "min_temp", "acc_precip"]
all_data = []

print("Fetching from DMI...")
for param in parameters:
    params = {
        "timeResolution": "day",
        "stationId": station_id,
        "datetime": f"{start_date}/{end_date}",
        "parameterId": param,
        "limit": 10000
    }
    
    response = requests.get(url, params=params)
    if response.status_code == 200:
        features = response.json().get('features', [])
        all_data.extend([f['properties'] for f in features])

df = pd.DataFrame(all_data)

if not df.empty and 'from' in df.columns:
    # Use string slicing to extract the date portion, bypassing .dt completely
    df['date'] = df['from'].astype(str).str[:10]
    
    # Pivot into wide format
    df_weather = df.pivot_table(
        index='date',
        columns='parameterId',
        values='value',
        aggfunc='first'
    ).reset_index()

    df_weather.columns.name = None
    df_weather = df_weather.rename(columns={
        "max_temp": "Max Temp (°C)", 
        "min_temp": "Min Temp (°C)", 
        "acc_precip": "Precipitation (mm)"
    })

    print("\n✅ Success:")
    print(df_weather.tail(10).to_string(index=False))
else:
    print("No data retrieved or 'from' key missing from response.")

Fetching from DMI...

✅ Success:
      date  Precipitation (mm)  Min Temp (°C)
2026-05-11                 0.2            5.6
2026-05-12                 3.4            6.2
2026-05-13                 7.7            6.5
2026-05-14                 0.0            4.9
2026-05-15                 0.4            6.8
2026-05-16                 5.9            6.8
2026-05-17                 0.2            7.2
2026-05-18                 3.5           10.9
2026-05-19                 1.6            9.9
2026-05-20                 1.8           11.8


In [6]:
import pandas as pd
import os

base_path = '/Users/fangsiyu/Desktop/GTFS_jau_feb/'

stop_times = pd.read_csv(os.path.join(base_path, 'stop_times.txt'))
trips = pd.read_csv(os.path.join(base_path, 'trips.txt'))
calendar = pd.read_csv(os.path.join(base_path, 'calendar.txt'))

kolding_times = stop_times[stop_times['stop_id'] == 8600083].copy()
m1 = pd.merge(kolding_times, trips, on='trip_id', how='left')
m2 = pd.merge(m1, calendar, on='service_id', how='left')

# 解碼行事曆：把抽象的「日期區間 + 星期幾」攤平成為「每一天真正的日期」
expanded_rows = []

for idx, row in m2.iterrows():
    # 修正語法：row 已經是 Series，直接用欄位名稱取值
    start_str = str(int(row['start_date']))
    end_str = str(int(row['end_date']))
    
    # 建立該班次營運範圍內的完整日期序列
    date_range = pd.date_range(start=start_str, end=end_str)
    
    # 收集這班車在星期幾有開 (0=週一, 1=週二 ... 6=週日)
    active_days = []
    if row['monday'] == 1: active_days.append(0)
    if row['tuesday'] == 1: active_days.append(1)
    if row['wednesday'] == 1: active_days.append(2)
    if row['thursday'] == 1: active_days.append(3)
    if row['friday'] == 1: active_days.append(4)
    if row['saturday'] == 1: active_days.append(5)
    if row['sunday'] == 1: active_days.append(6)
    
    # 過濾出符合星期規則的確切日期
    final_dates = date_range[date_range.weekday.isin(active_days)]
    
    # 把每一個確切日期拆成獨立的一行加入結果
    for d in final_dates:
        expanded_rows.append({
            'exact_date': d.strftime('%Y-%m-%d'),
            'trip_id': row['trip_id'],
            'arrival_time': row['arrival_time'],
            'departure_time': row['departure_time'],
            'trip_headsign': row['trip_headsign']
        })

daily_schedule = pd.DataFrame(expanded_rows)
# print(daily_schedule)

# 依日期與發車時間排序
daily_schedule = daily_schedule.sort_values(by=['exact_date', 'departure_time']).reset_index(drop=True)

# 7. 直接印出前 30 筆看結果
print(f"\n--- 成功解碼！已成功展開成『每日時刻表』（共 {len(daily_schedule)} 筆紀錄）---")
daily_schedule.head(30)

# output_file = os.path.join(base_path, 'K_daily_trains_jan_26th.csv')
# daily_schedule.to_csv('/Users/fangsiyu/Desktop/Kolding Hackathon 2026/K_daily_trains_jan_26th.csv', index=False, encoding='utf-8-sig')
# print(f"\n【重要提示】已將完整的每日明細表匯出至：{output_file}")

/var/folders/s0/f874234s21s4w3fp3rnc3rfm0000gn/T/ipykernel_99278/2289027365.py:6: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  stop_times = pd.read_csv(os.path.join(base_path, 'stop_times.txt'))



--- 成功解碼！已成功展開成『每日時刻表』（共 9989 筆紀錄）---


,exact_date,trip_id,arrival_time,departure_time,trip_headsign
0,2026-01-26,157435850,10:03:00,10:05:00,CPH Lufthavn
1,2026-01-26,157434212,10:04:00,10:06:00,Esbjerg St.
2,2026-01-26,157434506,10:11:00,10:13:00,Østerport St.
3,2026-01-26,157435575,10:40:00,10:43:00,Flensburg
4,2026-01-26,157434478,10:46:00,10:48:00,Esbjerg St.
5,2026-01-26,157434119,10:51:00,10:53:00,Skanderborg St.
6,2026-01-26,157434207,11:04:00,11:06:00,Esbjerg St.
7,2026-01-26,157434505,11:11:00,11:13:00,Østerport St.
8,2026-01-26,157435584,11:16:00,11:18:00,Fredericia St.
9,2026-01-26,157438496,11:44:00,11:46:00,København H


In [14]:
import pandas as pd
import os

# 1. 你的 1 月/2 月歷史資料解壓路徑
base_path = '/Users/fangsiyu/Desktop/GTFS/'

# 2. 讀取所有組裝所需的原始表格
print("正在讀取原始文字檔...")
stops = pd.read_csv(os.path.join(base_path, 'stops.txt'))
stop_times = pd.read_csv(os.path.join(base_path, 'stop_times.txt'))
trips = pd.read_csv(os.path.join(base_path, 'trips.txt'))
calendar = pd.read_csv(os.path.join(base_path, 'calendar.txt'))

# 3. 查證與模糊鎖定公車總站：把所有包含 'Kolding Busterminal' 或 'Mazantigade' 的站點 ID 全部撈出來
print("正在查證並過濾 Kolding 公車總站的所有月台 ID...")
bus_terminal_mask = (
    stops['stop_name'].str.contains('Kolding Busterminal', case=False, na=False)
)
kolding_bus_stops = stops[bus_terminal_mask][['stop_id', 'stop_name']]
bus_stop_ids = kolding_bus_stops['stop_id'].tolist()

print(f"查證成功！在資料庫中共綁定了 {len(bus_stop_ids)} 個相關的公車停靠點 ID。")


# 4. 利用這一群公車 ID，過濾出所有經過公車總站的時間點
kolding_bus_times = stop_times[stop_times['stop_id'].isin(bus_stop_ids)].copy()

# 5. 與車次、行事曆進行融合 (Join)
print("正在融合公車路線、車次與營運日曆...")
m1 = pd.merge(kolding_bus_times, trips, on='trip_id', how='left')
m2 = pd.merge(m1, calendar, on='service_id', how='left')

# 6. 解碼行事曆：把抽象的「日期區間 + 星期幾」攤平成為「每一天真正的日期」
print("正在展開公車排班規則，攤平成每日公車時刻表...")
expanded_rows = []

for idx, row in m2.iterrows():
    # 確保讀取整數日期的安全格式 (YYYYMMDD)
    start_str = str(int(row['start_date']))
    end_str = str(int(row['end_date']))
    
    # 建立該班次營運範圍內的完整日期序列
    date_range = pd.date_range(start=start_str, end=end_str)
    
    # 收集這班車在星期幾有開 (0=週一, 1=週二 ... 6=週日)
    active_days = []
    if row['monday'] == 1: active_days.append(0)
    if row['tuesday'] == 1: active_days.append(1)
    if row['wednesday'] == 1: active_days.append(2)
    if row['thursday'] == 1: active_days.append(3)
    if row['friday'] == 1: active_days.append(4)
    if row['saturday'] == 1: active_days.append(5)
    if row['sunday'] == 1: active_days.append(6)
    
    # 過濾出符合星期規則的確切日期
    final_dates = date_range[date_range.weekday.isin(active_days)]
    
    # 把每一個確切日期拆成獨立的一行加入結果
    for d in final_dates:
        expanded_rows.append({
            'exact_date': d.strftime('%Y-%m-%d'),
            'trip_id': row['trip_id'],
            'arrival_time': row['arrival_time'],
            'departure_time': row['departure_time'],
            'trip_headsign': row['trip_headsign'],
            'stop_id': row['stop_id']
        })

# 7. 建立最終的每日公車時刻表 DataFrame
daily_bus_schedule = pd.DataFrame(expanded_rows)

# 補回人類看得懂的公車站牌名稱 (如 Mazantigade v Kolding Busterminal)
daily_bus_schedule = pd.merge(daily_bus_schedule, kolding_bus_stops[['stop_id', 'stop_name']], on='stop_id', how='left')

# 時間格式優化：靠右補零 (例如 9:15:00 補成 09:15:00)，確保排序完美
daily_bus_schedule['arrival_time'] = daily_bus_schedule['arrival_time'].astype(str).str.rjust(8, '0')
daily_bus_schedule['departure_time'] = daily_bus_schedule['departure_time'].astype(str).str.rjust(8, '0')

# 依日期與發車時間排序
daily_bus_schedule = daily_bus_schedule.sort_values(by=['exact_date', 'departure_time']).reset_index(drop=True)

# 整理欄位順序
daily_bus_schedule = daily_bus_schedule[[
    'exact_date', 
    'stop_name', 
    'arrival_time', 
    'departure_time', 
    'trip_headsign', 
    'trip_id'
]]

print(f"\n--- 成功解碼！已成功展開成『公車總站每日時刻表』（共 {len(daily_bus_schedule)} 筆紀錄）---")
daily_bus_schedule.head(30)
daily_bus_schedule.to_csv('/Users/fangsiyu/Desktop/Kolding Hackathon 2026/K_daily_buses_th.csv', index=False, encoding='utf-8-sig')

正在讀取原始文字檔...


/var/folders/s0/f874234s21s4w3fp3rnc3rfm0000gn/T/ipykernel_99278/1882126750.py:10: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  stop_times = pd.read_csv(os.path.join(base_path, 'stop_times.txt'))


正在查證並過濾 Kolding 公車總站的所有月台 ID...
查證成功！在資料庫中共綁定了 6 個相關的公車停靠點 ID。
正在融合公車路線、車次與營運日曆...
正在展開公車排班規則，攤平成每日公車時刻表...

--- 成功解碼！已成功展開成『公車總站每日時刻表』（共 53874 筆紀錄）---
